Run this from the mother directory of DRAFT

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from dotenv import load_dotenv
import os
import torch
import json
import time

# Change working directory to DRAFT
os.chdir('DRAFT')
new_directory = os.getcwd()
print(f"Working directory: {new_directory}")

load_dotenv()
HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE")

Working directory: /research/phd/phd2k22/cse/rudra.dhar/DRAFT


In [2]:
# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

cache_dir = "../cache"
model_name = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir, token=HUGGINGFACE_TOKEN)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=cache_dir, dtype=torch.bfloat16, token=HUGGINGFACE_TOKEN, device_map="auto")

# Move the model to the chosen device
model.to(device)

# Set the model to evaluation mode
model.eval()

Using device: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwi

In [12]:
"""
Define helper functions
"""
def load_jsonl(file_path):
    """Load a JSONL file and return list of dicts."""
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
            
        
    return data

def save_jsonl(data, file_path):
    """Append list of dicts to a JSONL file."""
    with open(file_path, "a", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
        
    

def generate_response(model, tokenizer, messages, device, max_new_tokens=500):
    """Generate model response given messages."""
    formatted_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Tokenize
    inputs = tokenizer(formatted_chat, return_tensors="pt").to(device)
    input_length = inputs["input_ids"].shape[1]
    # Generate
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    generated_ids = outputs[0][input_length:]  # slice only new tokens
    response = tokenizer.decode(generated_ids, skip_special_tokens=True) # Decode only the generated text
    generated_tokens = generated_ids.shape[0] # Number of generated tokens
    return response, generated_tokens


def extract_context(entry):
    """Extract context string from one JSONL entry."""
    return entry["Anchor"]["Context"]

def extract_retrieved_context(entry):
    """Extract retrieved context string from one JSONL entry."""
    contexts = [doc["Context"] for doc in entry["Retrieved"]]
    return contexts

def extract_retrieved_decision(entry):
    """Extract retrieved decision string from one JSONL entry."""
    decisions = [doc["Decision"] for doc in entry["Retrieved"]]
    return decisions

def extract_title(entry):
    """Extract title from one JSONL entry."""
    return entry["Anchor"]["Title"]

def extract_retrieved_titles(entry):
    """Extract retrieved titles from one JSONL entry."""
    titles = [doc["Title"] for doc in entry["Retrieved"]]
    return titles

def extract_retrieved_body(entry):
    """Extract retrieved body string from one JSONL entry."""
    bodies = [doc["Body"] for doc in entry["Retrieved"]]
    return bodies

def extract_primary_key(entry):
    """Extract primary key from one JSONL entry."""
    return entry["Anchor"]["PrimaryKey"]

def context_formator(retrieved_contexts, retrieved_decisions, context):
    messages = [
        {"role": "system", "content": "You are an expert software architect responsible for maintaining and thoroughly documenting all architectural decisions. You are writing an Architectural Decision Record for a software. Below are a few examples of Context and the corresponding Decision. Following the examples, provide only the ## Decision for the final ## Context provided by the user. Provide only the Decision in about 2-400 words. Do not add any explanations, introductions, or additional responses."},
        {"role": "user", "content": f"## Context: {retrieved_contexts[0]}"},
        {"role": "assistant", "content": f"## Decision: {retrieved_decisions[0]}"},
        {"role": "user", "content": f"## Context: {retrieved_contexts[1]}"},
        {"role": "assistant", "content": f"## Decision: {retrieved_decisions[1]}"},
        {"role": "user", "content": f"## Context: {context}"},
    ]
    return messages

def title_formator(retrieved_title, retrieved_body, title):
    messages = [
        {"role": "system", "content": "You are an expert software architect responsible for maintaining and thoroughly documenting all architectural decisions. You are writing an Architectural Decision Record for a software. Below are a few examples of Title and the corresponding Body of an ADR. Following the examples, provide only the Body for the final # Title provided by the user. Provide only the ADR content in about 10-800 words. Do not add any additional responses—only the ADR content."},
        {"role": "user", "content": f"# {retrieved_title[0]}"},
        {"role": "assistant", "content": retrieved_body[0]},
        {"role": "user", "content": f"# {retrieved_title[1]}"},
        {"role": "assistant", "content": retrieved_body[1]},
        {"role": "user", "content": f"# {title}"}
    ]
    return messages


In [14]:
input_file = "Retrieval/TBtest.jsonl"
# output_file = "RAFG/Results/gemma-3-4b-it-CDtest-results.jsonl"
entries = load_jsonl(input_file)

entry = entries[0]
primary_key = extract_primary_key(entry)
context = extract_title(entry)
print(context)
retrieved_contexts = extract_retrieved_titles(entry)
print(retrieved_contexts)
retrieved_decisions = extract_retrieved_body(entry)
print(retrieved_decisions)
messages = title_formator(retrieved_contexts, retrieved_decisions, context)
messages

6. Fetch nomsNumber on demand on calls to custody endpoint
['16. Use the Elite2 API to access NOMIS data', 'API-006: Universal Namespace']
["\nDate: 2019-01-18\n\n## Status\n\nAccepted\n\n## Context\n\nOur main source of data on prisoners and prison staff is NOMIS. The\nallocation tool currently uses the Custody API, as decided in [ADR 0006](https://github.com/ministryofjustice/offender-management-architecture-decisions/blob/master/decisions/0006-use-the-custody-api-to-access-nomis-data.md), to retrieve\ndata on both offenders and staff.\n\nThere are still four APIs into NOMIS providing general data access, with varying\napproaches to presenting the data and authentication. We still do not want to add\nto this duplication.\n\nAlthough it has been agreed by the HMPPS technical community that we would\nlike to move all clients to use the Custody API in preference to the other APIs,\ninitial use of the Custody API has raised some issues. Problems exist\nwith the locality of data, N+1 API 

[{'role': 'system',
  'content': 'You are an expert software architect responsible for maintaining and thoroughly documenting all architectural decisions. You are writing an Architectural Decision Record for a software. Below are a few examples of Title and the corresponding Body of an ADR. Following the examples, provide only the Body for the final # Title provided by the user. Provide only the ADR content in about 10-800 words. Do not add any additional responses—only the ADR content.'},
 {'role': 'user', 'content': '# 16. Use the Elite2 API to access NOMIS data'},
 {'role': 'assistant',
  'content': "\nDate: 2019-01-18\n\n## Status\n\nAccepted\n\n## Context\n\nOur main source of data on prisoners and prison staff is NOMIS. The\nallocation tool currently uses the Custody API, as decided in [ADR 0006](https://github.com/ministryofjustice/offender-management-architecture-decisions/blob/master/decisions/0006-use-the-custody-api-to-access-nomis-data.md), to retrieve\ndata on both offende

### Context to Decision

In [ ]:
input_file = "Retrieval/CDtest.jsonl"
output_file = "RAFG/Results/gemma-3-4b-it-CDtest.jsonl"
entries = load_jsonl(input_file)

results = []

# Iterate over entries
for i, entry in enumerate(entries[:3]): # limit to first 3 for demo
    primary_key = extract_primary_key(entry)
    context = extract_context(entry)
    retrieved_contexts = extract_retrieved_context(entry)
    retrieved_decisions = extract_retrieved_decision(entry)
    messages = context_formator(retrieved_contexts, retrieved_decisions, context)

    start_time = time.time()
    response, gen_tokens = generate_response(model, tokenizer, messages, device)
    elapsed = time.time() - start_time


    result = {
    "PrimaryKey": primary_key,
    "Decision": response,
    "GeneratedTokens": gen_tokens,
    "Time": elapsed
    }
    results.append(result)

# Save all results to output JSONL
save_jsonl(results, output_file)
print(f"Results saved to {output_file}")

Results saved to RAFG/Results/gemma-3-4b-it-CDtest-results.jsonl


### Title to Body

In [ ]:
input_file = "Retrieval/TBtest.jsonl"
output_file = "RAFG/Results/gemma-3-4b-it-TBtest.jsonl"
entries = load_jsonl(input_file)

results = []

# Iterate over entries
for i, entry in enumerate(entries[:3]): # limit to first 3 for demo
    primary_key = extract_primary_key(entry)
    title = extract_title(entry)
    retrieved_titles = extract_retrieved_titles(entry)
    retrieved_bodies = extract_retrieved_body(entry)
    messages = title_formator(retrieved_titles, retrieved_bodies, title)

    start_time = time.time()
    response, gen_tokens = generate_response(model, tokenizer, messages, device, max_new_tokens=1000)
    elapsed = time.time() - start_time

    result = {
    "PrimaryKey": primary_key,
    "Body": response,
    "GeneratedTokens": gen_tokens,
    "Time": elapsed
    }
    results.append(result)

# Save all results to output JSONL
save_jsonl(results, output_file)
print(f"Results saved to {output_file}")

Results saved to Prompting/Results/gemma-3-4b-it-TBtest-results.jsonl
